
# 練習問題: 配列を手作業でスレッドに分割する

## 目標

work-sharing 構文 (`for` / `do`) はまだ学んでいない. ここでは `omp_get_thread_num()` と `omp_get_num_threads()` だけを使って, 配列を**手作業で**スレッドに分割し, 各スレッドが自分の担当範囲の部分和を計算・表示する. 次のトピックで学ぶ `for` 構文が, なぜ便利なのかを先取りして体感する.

## 課題

`manual_partition.cpp` (または `manual_partition.f90`) は, 要素数 `n = 100` の配列 `a` (`a[i] = i + 1`) を用意し, スレッド `t` (全 `T` スレッド) が範囲 `[t*n/T, (t+1)*n/T)` を担当して, その部分和を表示する.

コメント `TODO` の指示に従って **OpenMP の指示行を追加** し, このブロックを複数スレッドで実行させよ.

- C++: ブロック `{ ... }` の直前に `#pragma omp parallel` を1行加える. `tid`, `nt`, `lo`, `hi`, `s` はブロック内で宣言してあるので, スレッドごとに別々の変数になる.
- Fortran: ブロックを `!$omp parallel private(tid, nt, lo, hi, i, s)` と `!$omp end parallel` で囲む. これらの変数をスレッドごとに別々にするため `private` 節を付ける.

**注意:** 1つの共有変数に全スレッドの和を足し込んではならない (それは競合 (data race) になる. 総和をまとめる `reduction` は後のトピック). 各スレッドは**自分の**部分和だけを表示すること.

それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore manual_partition.cpp -o manual_partition.exe

# Fortran
nvfortran -fast -mp=multicore manual_partition.f90 -o manual_partition.exe
```

```
OMP_NUM_THREADS=4 ./manual_partition.exe
```

## 期待される結果

`OMP_NUM_THREADS` に指定した数だけ行が表示され, 各スレッドが重ならない範囲を担当する (順番は実行ごとに入れ替わってよい). 例 (4スレッド):

```
thread 0 of 4: range [0, 25), partial sum = 325.000000
thread 1 of 4: range [25, 50), partial sum = 950.000000
thread 2 of 4: range [50, 75), partial sum = 1575.000000
thread 3 of 4: range [75, 100), partial sum = 2200.000000
```

`OMP_NUM_THREADS` の値を変えると, 範囲の分割と表示行数が変わることを確認せよ. すべての部分和を足すと全体の和 (5050) になる.



# 1. AIチューター
- 以下は必要に応じて実行（毎度実行する必要はない）


In [ ]:
import heytutor


## 1-1. 一般的な質問
- ChatGPTなどに聞くときのように自由に質問可能。
- ただし「答えを教えて」などは自制すること。


In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 1-2. この問題に関するヒント
- `{file:problem.md}` は上記の問題文に展開される。


In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}


## 1-3. いくつかの変数
* それぞれ以下のように展開される。

* `{file:FILENAME}` : _FILENAME_ の中身
* `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前の入力・出力, etc.

## 1-4. 困ったときのヘルプ
* コンパイル時や実行時のエラー直後に以下を実行するとエラーに関するヘルプが得られる。


In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:manual_partition.cpp}

コマンドと実行結果:
{bash[-1]}



## 1-5. フィードバック
* 答えが出た後も、無駄なところや、より良いやり方がないかを聞くことを推奨。
* 以下のファイル名は適宜書き換えよ (Fortranなら `.cpp` -> `.f90` とするなど)


In [ ]:
%%hey

フィードバックを下さい。

問題:
{file:problem.md}

私の答:
{file:manual_partition.cpp}


# 2. ジョブ投入ツール
- 以下を実行しておくと、`%%bash_submit_a` (Aquariousへのジョブ投入), `%%bash_submit_o` (Odyssey へのジョブ投入) が使えるようになる


In [ ]:
import wisteria_submit

# 3. C++ ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ manual_partition.cpp
#include <cstdio>
#include <omp.h>

int main() {
  const int n = 100;
  double a[n];
  for (int i = 0; i < n; i++) {
    a[i] = i + 1;
  }
  // TODO: 下のブロックの直前に #pragma omp parallel を1行追加し, 各スレッドが自分の担当範囲を計算するようにせよ.
  {
    int tid = omp_get_thread_num();
    int nt  = omp_get_num_threads();
    int lo  = tid * n / nt;
    int hi  = (tid + 1) * n / nt;
    double s = 0.0;
    for (int i = lo; i < hi; i++) {
      s += a[i];
    }
    printf("thread %d of %d: range [%d, %d), partial sum = %f\n",
           tid, nt, lo, hi, s);
  }
  return 0;
}

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore manual_partition.cpp -o manual_partition_cpp.exe

## 3-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./manual_partition_cpp.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a

./manual_partition_cpp.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./manual_partition_cpp.exe

## 3-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:manual_partition.cpp}


# 4. Fortran ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ manual_partition.f90
program manual_partition
  use omp_lib
  integer, parameter :: n = 100
  real(8) :: a(0:n-1), s
  integer :: i, tid, nt, lo, hi
  do i = 0, n - 1
     a(i) = i + 1
  end do
  ! TODO: 下のブロックを !$omp parallel private(tid, nt, lo, hi, i, s) ... !$omp end parallel で囲み, 各スレッドが自分の担当範囲を計算するようにせよ.
  tid = omp_get_thread_num()
  nt  = omp_get_num_threads()
  lo  = tid * n / nt
  hi  = (tid + 1) * n / nt
  s = 0.0d0
  do i = lo, hi - 1
     s = s + a(i)
  end do
  print "(a,i0,a,i0,a,i0,a,i0,a,f0.6)", &
       "thread ", tid, " of ", nt, ": range [", lo, ", ", hi, "), partial sum = ", s
  ! TODO: 上で始めた parallel 領域を閉じる (!$omp end parallel).
end program manual_partition

## 4-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore manual_partition.f90 -o manual_partition_f90.exe

## 4-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./manual_partition_f90.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a
./manual_partition_f90.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit

#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./manual_partition_f90.exe

## 4-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:manual_partition.f90}